In [19]:
import requests

import numpy as np
import pandas as pd

import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [20]:
#Data Enrichment
#1.Read the concatinated data
#2.Handle Missing values
#3.Create derived columns for further analyssi
#4.Save the data in data/JC

In [25]:
#Read the concatinated data
citibike_df = pd.read_csv("../data/citibike/JC2025.csv")

#### Step 7: Setting Dates


In [26]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'], errors='coerce')
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'], errors='coerce')

In [27]:
# Ստուգլ, սյունակների տիպերը, Եթե այն ցույց է տալիս object, նշ․ է՝ to_datetime-ը չի կարողացել ոչինչ փոխել:
print(citibike_df[['started_at', 'ended_at']].dtypes)

started_at    datetime64[us]
ended_at      datetime64[us]
dtype: object


#### Step 8: df overview


In [28]:
citibike_df.info

<bound method DataFrame.info of                   ride_id  rideable_type              started_at  \
0        880A0159BA5275FB  electric_bike 2025-01-16 17:50:49.136   
1        1A5E1E274B2AF0AD  electric_bike 2025-01-31 06:10:41.818   
2        EA9928D3C05B8377   classic_bike 2025-01-09 16:42:50.213   
3        3C42C367750B9292  electric_bike 2025-01-21 16:14:14.398   
4        94D3B0265A7BDE1F   classic_bike 2025-01-30 16:38:18.840   
...                   ...            ...                     ...   
2005403  56D74535493E0013  electric_bike 2025-12-23 10:27:20.255   
2005404  78AF7D2BDF1F54A0   classic_bike 2025-12-20 11:13:22.773   
2005405  774E8E39F0D74488   classic_bike 2025-12-19 20:49:37.159   
2005406  DEAB98A7DAC4AC0E  electric_bike 2025-12-12 15:52:15.914   
2005407  B768B41D7B764802  electric_bike 2025-12-10 07:51:23.154   

                       ended_at                        start_station_name  \
0       2025-01-16 17:57:00.710                                   Hilltop 

In [29]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

#### Step 9. Handling the missig values

In [30]:
citibike_df.shape

(2005408, 13)

In [31]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,8794,0.004385
10,end_lat,6888,0.003435
11,end_lng,6888,0.003435
6,end_station_name,6470,0.003226
4,start_station_name,6,0.000003
5,start_station_id,6,0.000003
8,start_lat,4,0.000002
9,start_lng,4,0.000002
0,ride_id,0,0.000000
1,rideable_type,0,0.000000


#### Step 10: Ride Duration


In [32]:
#Now we create ride duration.
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

##### Removing extremes

In [33]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)


In [34]:
citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

#### Step 11: Time Based Variables


In [35]:
#For later aggregation, we create useful time columns.
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [36]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual,ride_duration_minutes,date,month,month_name,day_of_week,hour
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member,6.192900,2025-01-16,2025-01,January,Thursday,17
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member,11.461350,2025-01-31,2025-01,January,Friday,6
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,21.377617,2025-01-09,2025-01,January,Thursday,16
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,22.934333,2025-01-21,2025-01,January,Tuesday,16
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member,25.822100,2025-01-30,2025-01,January,Thursday,16


In [37]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [38]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-01-16 17:50:49.136,2025-01-16,2025-01,January,Thursday,17,Winter
1,2025-01-31 06:10:41.818,2025-01-31,2025-01,January,Friday,6,Winter
2,2025-01-09 16:42:50.213,2025-01-09,2025-01,January,Thursday,16,Winter
3,2025-01-21 16:14:14.398,2025-01-21,2025-01,January,Tuesday,16,Winter
4,2025-01-30 16:38:18.840,2025-01-30,2025-01,January,Thursday,16,Winter


#### Store the data

In [40]:
#Once we have created all the required derived columns we can save the data again as JC2025_Enriched.csv file
citibike_df.to_csv("../data/citibike/JC/JC2025_Enriched.csv", index = False)